# DIFUSCO — TSPTW-D Benchmark

TSP with Time Windows and Disruptions.
Compares DIFUSCO against classical baselines on the full problem.

| Section | Content |
|---------|--------|
| 1 | Load model |
| 2 | Load instances from datasets/ |
| 3 | Run benchmark (all sizes) |
| 4 | Results table |
| 5 | Denoising process visualisation |
| 6 | Step-by-step tour visualisation with decision rationale |

In [ ]:
import sys, os, time
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from data  import (load_dataset, random_instance, generate_time_windows,
                   generate_perturbations, build_tsptwd_features,
                   nn_tour_labels, two_opt_improve, greedy_decode,
                   tour_length, evaluate_tsptwd, time_aware_nn_tour)
from model import DifuscoModel, MODEL_SIZES
from train import NoiseSchedule, sample, sample_best_of, forward_diffuse

torch.manual_seed(42)
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## Section 1 — Load model

Train first if checkpoint does not exist:
```bash
python train.py --n 10 --epochs 3000 --size small --use_dataset
```

In [ ]:
SIZE = 'medium'
d, L, T = MODEL_SIZES[SIZE]

# TSPTW-D model: node_dim=5, edge_dim=2
model = DifuscoModel(d=d, L=L, node_dim=5, edge_dim=2)
schedule = NoiseSchedule(T=T)

ckpt = f'model/difusco_{SIZE}.pt'
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location='cpu'))
    print(f'Loaded {ckpt}')
else:
    print(f'WARNING: {ckpt} not found — using random weights')
model.eval()
model.to(DEVICE)
schedule.to(DEVICE)


## Section 2 — Load instances

In [ ]:
SIZES = [10, 20, 50, 100]
instances = {}
for n in SIZES:
    try:
        instances[n] = load_dataset(n)
        print(f'n={n:<5}  nodes={instances[n]["coords"].shape[0]}  perturbs={len(instances[n]["perturbations"])}')
    except FileNotFoundError as e:
        print(f'n={n}: {e}')

# Also add NN labels for evaluation
for n, inst in instances.items():
    inst['y0'] = nn_tour_labels(inst['coords'])

## Section 3 — Run benchmark

In [ ]:
def run_nn_tsptwd(inst):
    """Nearest-neighbour, ignores time windows."""
    dist = torch.cdist(inst['coords'], inst['coords'])
    p = 1.0 / (dist + 1e-8); p.fill_diagonal_(0.0)
    return greedy_decode(p, start=0)

def run_tw_nn(inst):
    """Time-window-aware nearest neighbour."""
    return time_aware_nn_tour(
        inst['coords'], inst['time_windows'],
        inst['service_times'], inst['perturbations'])

def run_difusco_tsptwd(inst, n_steps=50, n_samples=1):
    """DIFUSCO: full reverse chain + greedy decode."""
    x  = inst['node_feats']
    ef = inst['edge_feats']
    if n_samples == 1:
        p = sample(model, x, schedule, edge_extra=ef,
                   n_steps=n_steps, ddim=True, device=DEVICE)
        return greedy_decode(p.cpu(), start=0)
    else:
        tour, _, _ = sample_best_of(model, x, schedule, edge_extra=ef,
                                     n_samples=n_samples, n_steps=n_steps, device=DEVICE)
        return tour

def eval_tour(inst, tour):
    t, pen, obj = evaluate_tsptwd(
        inst['coords'], tour,
        inst['time_windows'], inst['service_times'], inst['perturbations'])
    return t, pen, obj


In [ ]:
print(f'{"n":<6} {"Method":<18} {"Travel":>8} {"Penalty":>10} {"Obj":>10} {"Time(ms)":>10}')
print('-' * 68)

for n, inst in instances.items():
    methods = [
        ('NN (no TW)',    run_nn_tsptwd),
        ('TW-aware NN',  run_tw_nn),
        ('DIFUSCO×1',    lambda i: run_difusco_tsptwd(i, n_samples=1)),
        ('DIFUSCO×8',    lambda i: run_difusco_tsptwd(i, n_samples=8)),
    ]
    for name, fn in methods:
        t0   = time.perf_counter()
        tour = fn(inst)
        ms   = (time.perf_counter() - t0) * 1000
        trav, pen, obj = eval_tour(inst, tour)
        print(f'{n:<6} {name:<18} {trav:>8.3f} {pen:>10.1f} {obj:>10.3f} {ms:>9.1f}ms')
    print()

## Section 5 — Denoising process visualisation

Shows the edge probability matrix at key timesteps during the reverse chain,
illustrating how DIFUSCO refines noise into a structured tour.

In [ ]:
inst = instances[10]
x    = inst['node_feats']
ef   = inst['edge_feats']
coords = inst['coords']
n    = coords.shape[0]

# Capture intermediate states
SNAPSHOTS = [500, 200, 100, 50, 20, 5, 1]  # timesteps to snapshot
snapshots = {}

with torch.no_grad():
    model.eval()
    dev = torch.device(DEVICE)
    xd  = x.to(dev); efd = ef.to(dev)
    T_  = schedule.T
    timesteps = torch.linspace(T_, 1, 50).long().tolist()

    y = torch.randn(n, n, device=dev)
    for idx, t in enumerate(timesteps):
        t_tensor = torch.tensor(t, dtype=torch.long, device=dev)
        y0_pred  = model(xd, y, t_tensor, edge_extra=efd)

        if t in SNAPSHOTS:
            snapshots[t] = y0_pred.cpu().detach()

        if idx == len(timesteps) - 1:
            y = y0_pred; break
        t_prev    = int(timesteps[idx + 1])
        ab_t      = schedule.alpha_bar(t).to(dev)
        ab_prev   = schedule.alpha_bar(t_prev).to(dev)
        noise_dir = (y - ab_t.sqrt() * y0_pred) / (1.0 - ab_t).clamp(min=1e-8).sqrt()
        y         = ab_prev.sqrt() * y0_pred + (1.0 - ab_prev).sqrt() * noise_dir

    snapshots['final'] = y.cpu()

# Plot
keys = SNAPSHOTS + ['final']
fig, axes = plt.subplots(1, len(keys), figsize=(2.2 * len(keys), 2.5))
for ax, k in zip(axes, keys):
    mat = snapshots[k].numpy()
    ax.imshow(mat, vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')
    ax.set_title(f't={k}' if k != 'final' else 'final ŷ₀', fontsize=8)
    ax.axis('off')
plt.suptitle('Denoising process — edge probability matrix', fontsize=10)
plt.tight_layout()
plt.savefig('figures/benchmark/tsptwd_denoising.png', dpi=150, bbox_inches='tight')
plt.show()
print('Each cell (i,j) = P(edge i→j in tour).  Red=0, Green=1.')

## Section 6 — Step-by-step tour visualisation

For each round of the greedy decode, shows:
- Which node was chosen and why (score vs runner-up)
- Unvisited nodes coloured by their DIFUSCO edge probability
- Arrival time and time-window status
- Whether the arc was perturbed

In [ ]:
LATE_TOL = 1.05

def visualize_difusco_rounds(inst, model, schedule, n_steps=50,
                              max_rounds=None, cols=3, figsize_per_cell=4.2):
    """
    Visualise DIFUSCO greedy decode step by step.

    1. Run full DDPM/DDIM reverse chain  →  p_hat (n,n)
    2. Replay greedy_decode step by step, tracking time and TW status
    3. Per subplot: nodes coloured by p_hat score, arrow shows chosen arc
    4. Title: chosen node, score delta vs runner-up, arrival time, TW status
    """
    coords  = inst['coords']
    tw      = inst['time_windows']
    svc     = inst['service_times']
    perturbs = inst['perturbations']
    n = coords.shape[0]

    # ── Run DIFUSCO inference ──────────────────────────────────────────────
    p_hat = sample(model, inst['node_feats'], schedule,
                   edge_extra=inst['edge_feats'],
                   n_steps=n_steps, ddim=True, device='cpu').detach()

    # ── Replay greedy decode ──────────────────────────────────────────────
    from torch import cdist as _cdist
    dist_mat = _cdist(coords, coords)

    steps   = []
    visited = torch.zeros(n, dtype=torch.bool)
    tour    = [0]; visited[0] = True
    t_sim   = 0.0

    for _ in range(n - 1):
        cur    = tour[-1]
        scores = p_hat[cur].clone().masked_fill(visited, float('-inf'))
        sorted_idx = scores.argsort(descending=True)
        chosen  = sorted_idx[0].item()
        top3    = [(j.item(), scores[j].item()) for j in sorted_idx[:3]]
        runner  = top3[1][1] if len(top3) > 1 else float('-inf')
        delta   = scores[chosen].item() - runner

        # Travel to chosen
        mult = 1.0
        for pi, pj, ts, te, alpha in perturbs:
            if (pi == cur and pj == chosen) or (pi == chosen and pj == cur):
                if ts <= t_sim <= te:
                    mult = max(mult, 1.0 + alpha)
        t_sim += dist_mat[cur, chosen].item() * mult

        a_i, b_i = tw[chosen, 0].item(), tw[chosen, 1].item()
        if t_sim < a_i:
            tw_status = 'early (waits)'; t_sim = a_i
        elif t_sim > b_i * LATE_TOL:
            tw_status = 'NEXT-DAY'
        elif t_sim > b_i:
            tw_status = 'LATE'
        else:
            tw_status = 'in window'
        t_sim += svc[chosen].item()

        steps.append(dict(cur=cur, chosen=chosen, score=scores[chosen].item(),
                          delta=delta, top3=top3, t_arrive=t_sim,
                          tw_status=tw_status, perturbed=mult > 1.0,
                          tour_so_far=list(tour)))
        tour.append(chosen); visited[chosen] = True

    if max_rounds:
        steps = steps[:max_rounds]

    # ── Plot ──────────────────────────────────────────────────────────────
    rows  = (len(steps) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols,
                             figsize=(cols * figsize_per_cell, rows * figsize_per_cell))
    axes = np.array(axes).reshape(-1)

    cmap  = cm.RdYlGn
    xy    = coords.numpy()

    all_scores = p_hat.numpy()
    vmin_g, vmax_g = all_scores.min(), all_scores.max()

    visited_set = {0}
    for step_idx, s in enumerate(steps):
        ax = axes[step_idx]

        # ── Draw tour so far (grey arrows) ──────────────────────────────
        for k in range(len(s['tour_so_far']) - 1):
            i, j = s['tour_so_far'][k], s['tour_so_far'][k+1]
            ax.annotate('', xy=xy[j], xytext=xy[i],
                        arrowprops=dict(arrowstyle='->', color='#aaaaaa', lw=1.0))

        # ── Draw new arc (red if perturbed, blue otherwise) ─────────────
        arc_color = 'red' if s['perturbed'] else 'royalblue'
        ax.annotate('', xy=xy[s['chosen']], xytext=xy[s['cur']],
                    arrowprops=dict(arrowstyle='->', color=arc_color, lw=2.2))

        # ── Node colours ────────────────────────────────────────────────
        for i in range(n):
            score_i = p_hat[s['cur'], i].item()
            if i in visited_set:
                c = '#cccccc'
            else:
                c = cmap((score_i - vmin_g) / max(vmax_g - vmin_g, 1e-8))
            marker = '*' if i == s['cur'] else ('D' if i == s['chosen'] else 'o')
            size   = 200 if marker != 'o' else 80
            ax.scatter(*xy[i], marker=marker, s=size, c=[c],
                       edgecolors='black', lw=0.8, zorder=5)
            if n <= 25:
                ax.text(*xy[i], str(i), ha='center', va='center',
                        fontsize=6, zorder=6)

        visited_set.add(s['chosen'])

        # ── Title ────────────────────────────────────────────────────────
        k    = step_idx + 1
        t3   = '  |  '.join(f'{j}:{sc:.2f}' for j, sc in s['top3'])
        pert = '  \u26a0 PERTURBED ARC' if s['perturbed'] else ''
        title = (f'Round {k}: {s["cur"]}\u2192{s["chosen"]}  '
                 f'(score {s["score"]:.3f})  (\u0394={s["delta"]:+.3f})\n'
                 f'Top-3: {t3}\n'
                 f't={s["t_arrive"]:.3f}  [{s["tw_status"]}]{pert}')
        ax.set_title(title, fontsize=7, pad=4)
        ax.set_xticks([]); ax.set_yticks([])

    # Hide unused axes
    for ax in axes[len(steps):]:
        ax.axis('off')

    # Shared colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap,
                                norm=mcolors.Normalize(vmin=vmin_g, vmax=vmax_g))
    sm.set_array([])
    fig.colorbar(sm, ax=axes.tolist(), shrink=0.6, label='DIFUSCO edge score')

    plt.suptitle('DIFUSCO TSPTW-D — round-by-round decode', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/benchmark/tsptwd_rounds.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
visualize_difusco_rounds(instances[10], model, schedule, n_steps=50, cols=3)

In [ ]:
# Larger instances — show first 12 rounds
# visualize_difusco_rounds(instances[50],  model, schedule, n_steps=50, max_rounds=12, cols=4)
# visualize_difusco_rounds(instances[100], model, schedule, n_steps=50, max_rounds=12, cols=4)